## **Librerias**

In [1]:
import math
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import timedelta

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

## **Cargar artefactos base y nuevos datos**

In [2]:
# Paths
PATH_BASE = "../data/raw/dataset_local.csv"          # histórico original
PATH_NEW  = "../data/raw/dataset_2022.csv"  # nuevos registros
PATH_FINAL = "../data/processed/dataset_final_retrain.csv"

PATH_MODEL = "../models/modelo_lstm_optimo.h5"
PATH_SCALERS = "../models/scalers_lstm_optimo.save"
PATH_FEATURES = "../models/features_lstm_optimo.save"

# Cargar datasets
df_base = pd.read_csv(PATH_BASE, parse_dates=["date"])
df_new  = pd.read_csv(PATH_NEW, parse_dates=["date"])

print("Base shape:", df_base.shape)
print("Nuevos shape:", df_new.shape)

display(df_base.head())
display(df_new.head())

Base shape: (1800, 11)
Nuevos shape: (2190, 11)


,product_id,date,product_name,quantity_sold,stock_available,replenishment,price_per_unit,total_sales_value,season,holiday,demand_status
0,P001M,2021-01-01,Manzana Roja,82,217,50,1.10,90.20,lluvioso,False,normal
1,P001M,2021-01-02,Manzana Roja,59,237,47,1.06,62.54,lluvioso,False,bajo
2,P001M,2021-01-03,Manzana Roja,79,201,31,1.13,89.27,lluvioso,False,normal
3,P001M,2021-01-04,Manzana Roja,67,232,50,1.13,75.71,lluvioso,False,normal
4,P001M,2021-01-05,Manzana Roja,93,179,23,0.90,83.70,lluvioso,False,normal


,product_id,date,product_name,quantity_sold,stock_available,replenishment,price_per_unit,total_sales_value,season,holiday,demand_status
0,P001M,2022-01-01,Manzana Roja,45,440,0,1.13,50.89,lluvioso,False,bajo
1,P001M,2022-01-02,Manzana Roja,95,345,0,1.13,107.43,lluvioso,False,normal
2,P001M,2022-01-03,Manzana Roja,75,270,0,1.13,84.81,lluvioso,False,normal
3,P001M,2022-01-04,Manzana Roja,51,219,0,1.13,57.67,lluvioso,False,bajo
4,P001M,2022-01-05,Manzana Roja,51,500,332,1.13,57.67,lluvioso,False,bajo


## **Validación de nuevos registros**

In [4]:
def validate_new_data(df_base, df_new):
    
    # Eliminar columnas basura si existen
    df_base = df_base.copy()
    df_new = df_new.copy()
    
    if "Unnamed: 0" in df_base.columns:
        df_base.drop(columns=["Unnamed: 0"], inplace=True)

    if "Unnamed: 0" in df_new.columns:
        df_new.drop(columns=["Unnamed: 0"], inplace=True)

    # 1. Columnas consistentes
    base_cols = set(df_base.columns)
    new_cols  = set(df_new.columns)

    missing = base_cols - new_cols
    extra   = new_cols - base_cols

    if missing:
        raise ValueError(f"Columnas faltantes en nuevos datos: {missing}")
    if extra:
        print(f"[WARN] Columnas extra en nuevos datos (se ignorarán): {extra}")
        df_new = df_new[df_base.columns]

    # 2. Fechas no nulas
    if df_new["date"].isna().any():
        raise ValueError("Hay fechas vacías en el dataset nuevo.")

    # 3. Duplicados (product_id, date)
    base_keys = set(zip(df_base["product_id"], df_base["date"]))
    new_keys  = set(zip(df_new["product_id"], df_new["date"]))

    duplicates = base_keys & new_keys
    if duplicates:
        print(f"[INFO] Se encontraron duplicados (product_id, date) que serán ignorados.")
        # filtrar nuevos sin duplicados existentes
        df_new = df_new[~list(zip(df_new["product_id"], df_new["date"])).__contains__]

    # 4. Valores negativos en columnas clave
    for col in ["quantity_sold", "stock_available", "replenishment", "price_per_unit"]:
        if col in df_new.columns and (df_new[col] < 0).any():
            raise ValueError(f"Hay valores negativos en {col} en nuevos datos.")

    return df_new

df_new_valid = validate_new_data(df_base, df_new)
print("Nuevos registros válidos:", df_new_valid.shape)

Nuevos registros válidos: (2190, 11)


## **Merge + recomputar features**

In [5]:
# Reutilizamos lo que ya hiciste en el Notebook A
def add_calendar_features(df):
    df = df.copy()
    df["day_of_week"] = df["date"].dt.dayofweek
    df["month"] = df["date"].dt.month
    df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)
    df["day_of_year"] = df["date"].dt.dayofyear
    df["week"] = df["date"].dt.isocalendar().week
    return df

def add_sales_features(df):
    df = df.copy()
    df["sales_mavg_7d"] = (
        df.groupby("product_id")["quantity_sold"]
        .transform(lambda x: x.rolling(7, min_periods=1).mean())
    )
    df["ratio_sold_stock"] = df["quantity_sold"] / (df["stock_available"] + 1)
    df["price_var"] = df["price_per_unit"] / \
                      df.groupby("product_id")["price_per_unit"].transform("mean")
    return df

# 1) Unir base + nuevos válidos
df_all = pd.concat([df_base, df_new_valid], ignore_index=True)
df_all = df_all.sort_values(["product_id", "date"]).reset_index(drop=True)

print("Shape combinado:", df_all.shape)

# 2) Recalcular features
df_all = add_calendar_features(df_all)
df_all = add_sales_features(df_all)

selected_features = [
    "product_id", 
    "date",
    "quantity_sold",
    "stock_available",
    "sales_mavg_7d",
    "ratio_sold_stock",
    "price_per_unit",
    "day_of_week",
    "month",
    "is_weekend"
]

df_model = df_all[selected_features].copy()
df_model.to_csv(PATH_FINAL, index=False)
print("Dataset para re-train guardado en:", PATH_FINAL)

Shape combinado: (3990, 11)
Dataset para re-train guardado en: ../data/processed/dataset_final_retrain.csv


## **Re-escalar y genera secuencias**

In [6]:
# Cargar lista de features que el modelo espera
features = joblib.load(PATH_FEATURES)
print("Features usadas por el modelo:", features)

def scale_per_product(df, feature_cols):
    scalers = {}
    scaled_groups = []

    for pid, group in df.groupby("product_id"):
        scaler = MinMaxScaler()
        scaled_values = scaler.fit_transform(group[feature_cols])

        scaled_group = pd.DataFrame(scaled_values, columns=feature_cols)
        scaled_group["product_id"] = pid
        scaled_group["date"] = group["date"].values

        scalers[pid] = scaler
        scaled_groups.append(scaled_group)

    df_scaled = pd.concat(scaled_groups).sort_values(["product_id", "date"])
    df_scaled.reset_index(drop=True, inplace=True)

    return df_scaled, scalers

df_retrain = pd.read_csv(PATH_FINAL, parse_dates=["date"]).sort_values(["product_id", "date"])
df_scaled_new, scalers_new = scale_per_product(df_retrain, features)

def generate_sequences(df, feature_cols, seq_len):
    X, y = [], []

    for pid, group in df.groupby("product_id"):
        data = group[feature_cols].values
        for i in range(len(data) - seq_len):
            X.append(data[i:i+seq_len])
            y.append(data[i+seq_len, 0])  # quantity_sold

    return np.array(X), np.array(y)

SEQ_LEN = 10
X_all, y_all = generate_sequences(df_scaled_new, features, SEQ_LEN)
print("Secuencias para re-train:", X_all.shape, y_all.shape)

Features usadas por el modelo: ['quantity_sold', 'stock_available', 'sales_mavg_7d', 'ratio_sold_stock', 'price_per_unit']
Secuencias para re-train: (3930, 10, 5) (3930,)


## **Re-entrenar modelo (antes vs despues)**

In [7]:
# Cargar modelo original DOS VECES:
model_old = load_model(PATH_MODEL, compile=False)
model_new = load_model(PATH_MODEL, compile=False)

model_new.compile(optimizer="adam", loss="mse", metrics=["mae"])

# Split temporal sencillo: usamos la cola para test
n = len(X_all)
test_size = int(n * 0.2)
X_train_rt, X_test_rt = X_all[:-test_size], X_all[-test_size:]
y_train_rt, y_test_rt = y_all[:-test_size], y_all[-test_size:]

print("Train:", X_train_rt.shape, "Test:", X_test_rt.shape)

es = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1)

history_rt = model_new.fit(
    X_train_rt, y_train_rt,
    epochs=8,
    batch_size=16,
    validation_split=0.1,
    shuffle=False,
    callbacks=[es, rlrop],
    verbose=1
)


Train: (3144, 10, 5) Test: (786, 10, 5)
Epoch 1/8
177/177 [==============================] - 3s 10ms/step - loss: 434.8694 - mae: 6.0306 - val_loss: 0.0237 - val_mae: 0.1313 - lr: 0.0010
Epoch 2/8
177/177 [==============================] - 1s 8ms/step - loss: 0.0385 - mae: 0.1590 - val_loss: 0.0235 - val_mae: 0.1306 - lr: 0.0010
Epoch 3/8
177/177 [==============================] - 2s 9ms/step - loss: 0.0367 - mae: 0.1558 - val_loss: 0.0239 - val_mae: 0.1319 - lr: 0.0010
Epoch 4/8
174/177 [============================>.] - ETA: 0s - loss: 0.0365 - mae: 0.1547
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
177/177 [==============================] - 2s 10ms/step - loss: 0.0364 - mae: 0.1545 - val_loss: 0.0247 - val_mae: 0.1333 - lr: 0.0010
Epoch 5/8
177/177 [==============================] - 2s 9ms/step - loss: 0.0356 - mae: 0.1534 - val_loss: 0.0274 - val_mae: 0.1399 - lr: 5.0000e-04


## **Evaluar antes vs despues (MAE/RMSE reales)**

In [8]:
# Usamos cualquier scaler como referencia para desescalar
pid_ref = list(scalers_new.keys())[0]
scaler_ref = scalers_new[pid_ref]
n_features = len(features)

def invert_scale(y_scaled, scaler, n_features):
    temp = np.zeros((len(y_scaled), n_features))
    temp[:, 0] = y_scaled
    return scaler.inverse_transform(temp)[:, 0]

# Predicciones antes (modelo viejo)
y_pred_old_scaled = model_old.predict(X_test_rt)
y_pred_new_scaled = model_new.predict(X_test_rt)

y_test_real = invert_scale(y_test_rt, scaler_ref, n_features)
y_pred_old_real = invert_scale(y_pred_old_scaled.flatten(), scaler_ref, n_features)
y_pred_new_real = invert_scale(y_pred_new_scaled.flatten(), scaler_ref, n_features)

def metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

mae_old, rmse_old = metrics(y_test_real, y_pred_old_real)
mae_new, rmse_new = metrics(y_test_real, y_pred_new_real)

print(f"Modelo original -> MAE: {mae_old:.2f}, RMSE: {rmse_old:.2f}")
print(f"Modelo re-train -> MAE: {mae_new:.2f}, RMSE: {rmse_new:.2f}")

25/25 [==============================] - 0s 5ms/step
Modelo original -> MAE: 9625.00, RMSE: 9625.03
Modelo re-train -> MAE: 22.55, RMSE: 28.12


**Si el nuevo es mejor**

In [10]:
if rmse_new <= rmse_old:
    print("El modelo re-entrenado mejora o iguala el desempeño. Guardando artefactos...")
    model_new.save(PATH_MODEL, include_optimizer=False)
    joblib.dump(scalers_new, PATH_SCALERS)
else:
    print("El modelo re-entrenado es peor. Se mantiene el modelo original.")

El modelo re-entrenado mejora o iguala el desempeño. Guardando artefactos...


/opt/miniconda3/envs/stock-ml/lib/python3.10/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


## **Comprobar que las predicciones cambiaron**

In [11]:
# Escogemos un producto y una fecha futura
product_test = df_retrain["product_id"].iloc[0]
future_date = (df_retrain["date"].max() + timedelta(days=30)).strftime("%Y-%m-%d")

print("Producto:", product_test, "Fecha futura:", future_date)

def predict_to_date_local(df, model, scalers, features, product_id, target_date, seq_len=10):
    df_p = df[df["product_id"] == product_id].sort_values("date")
    if len(df_p) < seq_len:
        return None

    scaler = scalers[product_id]
    recent = df_p[features].tail(seq_len).values
    input_seq = scaler.transform(recent)

    last_date = df_p["date"].max()
    target_date = pd.to_datetime(target_date)
    steps = (target_date - last_date).days

    if steps <= 0:
        return None

    pred = None
    for _ in range(steps):
        pred = model.predict(input_seq[np.newaxis, :, :], verbose=0)[0][0]
        new_row = input_seq[-1].copy()
        new_row[0] = pred
        input_seq = np.vstack([input_seq[1:], new_row])

    temp = np.zeros((1, len(features)))
    temp[0, 0] = pred
    pred_real = scaler.inverse_transform(temp)[0, 0]
    return float(pred_real)

pred_old = predict_to_date_local(df_retrain, model_old, scalers_new, features, product_test, future_date)
pred_new = predict_to_date_local(df_retrain, model_new, scalers_new, features, product_test, future_date)

print(f"Predicción ANTES del re-train: {pred_old:.2f}")
print(f"Predicción DESPUÉS del re-train: {pred_new:.2f}")

Producto: P001M Fecha futura: 2023-01-30


/opt/miniconda3/envs/stock-ml/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/stock-ml/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Predicción ANTES del re-train: 9731.85
Predicción DESPUÉS del re-train: 73.09
